### LIBRARY IMPORTS

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.metrics import accuracy_score

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.path.dirname(os.getcwd()).endswith('notebooks'):
    os.chdir('../..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor

### CONFIGURATION

In [3]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

y_test = np.array(test.dataset.targets)[test.indices]

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

test = processor.convert_to_numpy(test) 

### DEFINE CNN

In [10]:
class CNN(CNNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(self, X_train, y_train, X_valid, y_valid, patience=10):
        
        in_channels = X_train.shape[1]
        output_size = int(np.max(y_train)) + 1

        image_size = X_train.shape[-1]

        conv1_out = image_size - self.kernel_size + 1
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - self.kernel_size + 1
        pool2_out = conv2_out // self.pool_size

        linear_input = self.channels[1] * pool2_out * pool2_out
        
        self._get_network(in_channels, linear_input, output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch: {epoch + 1} | Validation Log Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

### BIG CNN

**Total parameters:** 156.810

In [9]:
big_cnn = CNN(epochs=100, learning_rate=0.0001, channels=[32, 64], kernel_size=5, pool_size=2, hidden_size=64, batch_size=32)
big_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = big_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.6555 | Validation Accuracy: 0.4144
Epoch: 2 | Validation Log Loss: 1.5418 | Validation Accuracy: 0.4486
Epoch: 3 | Validation Log Loss: 1.5083 | Validation Accuracy: 0.4651
Epoch: 4 | Validation Log Loss: 1.4261 | Validation Accuracy: 0.4947
Epoch: 5 | Validation Log Loss: 1.3508 | Validation Accuracy: 0.5224
Epoch: 6 | Validation Log Loss: 1.3052 | Validation Accuracy: 0.5451
Epoch: 7 | Validation Log Loss: 1.2859 | Validation Accuracy: 0.5486
Epoch: 8 | Validation Log Loss: 1.2474 | Validation Accuracy: 0.5613
Epoch: 9 | Validation Log Loss: 1.2161 | Validation Accuracy: 0.5744
Epoch: 10 | Validation Log Loss: 1.2139 | Validation Accuracy: 0.5793
Epoch: 11 | Validation Log Loss: 1.1868 | Validation Accuracy: 0.5835
Epoch: 12 | Validation Log Loss: 1.1762 | Validation Accuracy: 0.5899
Epoch: 13 | Validation Log Loss: 1.1506 | Validation Accuracy: 0.5968
Epoch: 14 | Validation Log Loss: 1.1369 | Validation Accuracy: 0.6034
Epoch: 15 | Validation Log Lo

### SMALL CNN

**Total parameters:** 10.410

In [11]:
small_cnn = CNN(epochs=100, learning_rate=0.001, channels=[8, 16], kernel_size=5, pool_size=2, hidden_size=16, batch_size=32)
small_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = small_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.6484 | Validation Accuracy: 0.3952
Epoch: 2 | Validation Log Loss: 1.5156 | Validation Accuracy: 0.4418
Epoch: 3 | Validation Log Loss: 1.4549 | Validation Accuracy: 0.4665
Epoch: 4 | Validation Log Loss: 1.4193 | Validation Accuracy: 0.4771
Epoch: 5 | Validation Log Loss: 1.3617 | Validation Accuracy: 0.5090
Epoch: 6 | Validation Log Loss: 1.3552 | Validation Accuracy: 0.5060
Epoch: 7 | Validation Log Loss: 1.3817 | Validation Accuracy: 0.4979
Epoch: 8 | Validation Log Loss: 1.3671 | Validation Accuracy: 0.5085
Epoch: 9 | Validation Log Loss: 1.2640 | Validation Accuracy: 0.5437
Epoch: 10 | Validation Log Loss: 1.3145 | Validation Accuracy: 0.5270
Epoch: 11 | Validation Log Loss: 1.2680 | Validation Accuracy: 0.5399
Epoch: 12 | Validation Log Loss: 1.2576 | Validation Accuracy: 0.5492
Epoch: 13 | Validation Log Loss: 1.2342 | Validation Accuracy: 0.5583
Epoch: 14 | Validation Log Loss: 1.2682 | Validation Accuracy: 0.5487
Epoch: 15 | Validation Log Lo